In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
df=pd.read_csv("../data/processed/emscad_cleaned.csv")
print(df.head())

                                       title  \
0                           Marketing Intern   
1  Customer Service - Cloud Video Production   
2    Commissioning Machinery Assistant (CMA)   
3          Account Executive - Washington DC   
4                        Bill Review Manager   

                                     company_profile  \
0  We're Food52, and we've created a groundbreaki...   
1  90 Seconds, the worlds Cloud Video Production ...   
2  Valor Services provides Workforce Solutions th...   
3  Our passion for improving quality of life thro...   
4  SpotSource Solutions LLC is a Global Human Cap...   

                                         description  \
0  Food52, a fast-growing, James Beard Award-winn...   
1  Organised - Focused - Vibrant - Awesome!Do you...   
2  Our client, located in Houston, is actively se...   
3  THE COMPANY: ESRI – Environmental Systems Rese...   
4  JOB TITLE: Itemization Review ManagerLOCATION:...   

                                     

In [2]:
X=df.drop('fraudulent',axis=1)
y=df['fraudulent']

In [3]:
text_cols=['title','company_profile','description','requirements','benefits',]
cate_cols=['employment_type','required_experience','required_education','industry','function','country']
binary_cols=['telecommuting','has_company_logo','has_questions','salary_missing','location_missing']
target='fraudulent'

In [4]:
for col in text_cols:
    X[col] = X[col].fillna('')

In [5]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [6]:
text_transformers = []
for column in text_cols:
    text_transformers.append(
        (column, TfidfVectorizer(max_features=3000, ngram_range=(1,2)), column)
    )

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [7]:
preprocessor = ColumnTransformer(
    transformers=[

        *text_transformers,

        (
            "categorical",
            categorical_transformer,
            cate_cols
        ),

        (
            "binary",
            "passthrough",
            binary_cols
        )

    ]
)

#df[cate_cols].isna().sum()

In [8]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(X_train_transformed.shape)
print(X_test_transformed.shape)

feature_names = preprocessor.get_feature_names_out()
print(len(feature_names))  # should equal 15290

(14304, 15290)
(3576, 15290)
15290


In [9]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_transformed, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", pd.Series(y_train_resampled).value_counts().to_dict())
print(X_train_resampled.shape)

Before SMOTE: {0: 13611, 1: 693}
After SMOTE: {0: 13611, 1: 13611}
(27222, 15290)


In [10]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(preprocessor, "../models/preprocessor.joblib")

feature_names = preprocessor.get_feature_names_out()
joblib.dump(feature_names, "../models/feature_names.joblib")

['../models/feature_names.joblib']

In [11]:
from scipy import sparse

sparse.save_npz("../data/processed/X_train_resampled.npz", X_train_resampled)
sparse.save_npz("../data/processed/X_test_transformed.npz", X_test_transformed)
y_train_resampled.to_csv("../data/processed/y_train_resampled.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)